# Diffusion_Experiment4_Compare

Loads the results produced by the three training-strategy notebooks:

* `Diffusion_Experiment1_Backprop.ipynb`
* `Diffusion_Experiment2_Hybrid.ipynb`
* `Diffusion_Experiment3_Hebbian.ipynb`

and compares them side by side: overlaid loss curves, generated-sample grids, and a summary table. This notebook does **not** retrain anything — it only reads the artifacts (`loss_history.json`, `generated_samples.png`, `trained_diffusion_model.pth`) each of the three notebooks already saved to `LOCAL_OUTPUT_DIR` (and optionally S3).

This is a lightweight CPU notebook — no GPU / DiffUNet needed here, just plotting and file I/O. `ml.t3.medium` is enough on SageMaker.

# 1. Imports

In [1]:
import os
import json

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


# 2. Configuration

`SOURCE` controls where results are pulled from:

* `"local"` — reads directly from `./outputs/<experiment_name>/` on this Studio instance. Only   works if this notebook runs on the *same* SageMaker Studio storage as the three training   notebooks (default for a single Studio user/space).
* `"s3"` — downloads each experiment's artifacts from S3 first. Use this if the three notebooks   ran on separate instances, or if you just want a clean, reproducible source of truth. Requires   `S3_BUCKET` to have been set (non-`None`) when those notebooks ran.

Experiment names and the `S3_PREFIX` pattern must match Section 2 of the three training notebooks.

In [2]:
SOURCE = "local"  # "local" or "s3"

EXPERIMENTS = [
    "Diffusion_Experiment1_Backprop",
    "Diffusion_Experiment2_Hybrid",
    "Diffusion_Experiment3_Hebbian",
]

LABELS = {
    "Diffusion_Experiment1_Backprop": "Backprop",
    "Diffusion_Experiment2_Hybrid": "Backprop + Hebbian",
    "Diffusion_Experiment3_Hebbian": "Hebbian only",
}

# Must match LOCAL_OUTPUT_DIR / S3_BUCKET / S3_PREFIX in each training notebook's Section 2
LOCAL_OUTPUT_ROOT = os.path.join(os.getcwd(), "outputs")
S3_BUCKET = None          # e.g. "my-diffusion-hebbian-experiments"
S3_PREFIX_TEMPLATE = "diffusion-hebbian/{experiment_name}"

# Where this notebook's own downloaded/cached copies and comparison outputs go
COMPARE_OUTPUT_DIR = os.path.join(LOCAL_OUTPUT_ROOT, "comparison")
os.makedirs(COMPARE_OUTPUT_DIR, exist_ok=True)

print(f"Source: {SOURCE}")
print(f"Comparing: {EXPERIMENTS}")
print(f"Comparison outputs will be written to: {COMPARE_OUTPUT_DIR}")


Source: local
Comparing: ['Diffusion_Experiment1_Backprop', 'Diffusion_Experiment2_Hybrid', 'Diffusion_Experiment3_Hebbian']
Comparison outputs will be written to: /mnt/custom-file-systems/s3/shared/outputs/comparison


# 3. Load Results

For `SOURCE = "s3"`, each experiment's `loss_history.json`, `generated_samples.png`, and `trained_diffusion_model.pth` are downloaded into a local cache directory that mirrors the training notebooks' layout, so the rest of this notebook can treat both sources identically.

In [3]:
def experiment_dir(experiment_name):
    """Return the local directory holding this experiment's artifacts, downloading from S3 first if needed."""
    local_dir = os.path.join(LOCAL_OUTPUT_ROOT, experiment_name)

    if SOURCE == "local":
        if not os.path.isdir(local_dir):
            raise FileNotFoundError(
                f"No local results found at {local_dir}. "
                f"Run {experiment_name}.ipynb on this instance first, or set SOURCE = 's3'."
            )
        return local_dir

    elif SOURCE == "s3":
        if not S3_BUCKET:
            raise ValueError("SOURCE is 's3' but S3_BUCKET is None — set it in Section 2.")

        os.makedirs(local_dir, exist_ok=True)
        s3 = boto3.client("s3")
        prefix = S3_PREFIX_TEMPLATE.format(experiment_name=experiment_name).rstrip("/")

        for filename in ["loss_history.json", "generated_samples.png", "trained_diffusion_model.pth"]:
            key = f"{prefix}/{filename}"
            local_path = os.path.join(local_dir, filename)
            try:
                s3.download_file(S3_BUCKET, key, local_path)
            except Exception as e:
                print(f"  (skipping {key}: {e})")

        return local_dir

    else:
        raise ValueError(f"Unknown SOURCE: {SOURCE!r}")


results = {}

for experiment_name in EXPERIMENTS:
    print(f"Loading {experiment_name} ...")
    exp_dir = experiment_dir(experiment_name)

    with open(os.path.join(exp_dir, "loss_history.json")) as f:
        loss_history = json.load(f)

    results[experiment_name] = {
        "dir": exp_dir,
        "loss_history": loss_history,
        "samples_path": os.path.join(exp_dir, "generated_samples.png"),
    }

print("\nLoaded:", list(results.keys()))


Loading Diffusion_Experiment1_Backprop ...
Loading Diffusion_Experiment2_Hybrid ...
Loading Diffusion_Experiment3_Hebbian ...

Loaded: ['Diffusion_Experiment1_Backprop', 'Diffusion_Experiment2_Hybrid', 'Diffusion_Experiment3_Hebbian']


# 4. Compare Loss Curves

Overlays all three training-loss curves on one plot. Since MNIST subset size, batch size, and epoch count are identical across the three training notebooks, the curves are directly comparable epoch-for-epoch.

In [4]:
plt.figure(figsize=(8, 5))

for experiment_name in EXPERIMENTS:
    loss_history = results[experiment_name]["loss_history"]
    plt.plot(
        range(1, len(loss_history) + 1),
        loss_history,
        marker="o",
        markersize=3,
        label=LABELS.get(experiment_name, experiment_name)
    )

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss — Backprop vs. Backprop+Hebbian vs. Hebbian-only")
plt.legend()
plt.grid(alpha=0.3)

compare_loss_path = os.path.join(COMPARE_OUTPUT_DIR, "loss_comparison.png")
plt.savefig(compare_loss_path, bbox_inches="tight", dpi=150)
plt.show()

print(f"Saved comparison plot to {compare_loss_path}")


Saved comparison plot to /mnt/custom-file-systems/s3/shared/outputs/comparison/loss_comparison.png


# 5. Compare Generated Samples

Stacks each experiment's `generated_samples.png` (produced in Section 12 of the corresponding training notebook) into a single figure, one row per training strategy, for a quick visual gut check alongside the loss numbers.

In [5]:
fig, axes = plt.subplots(len(EXPERIMENTS), 1, figsize=(10, 3 * len(EXPERIMENTS)))

if len(EXPERIMENTS) == 1:
    axes = [axes]

for ax, experiment_name in zip(axes, EXPERIMENTS):
    samples_path = results[experiment_name]["samples_path"]
    if os.path.exists(samples_path):
        img = mpimg.imread(samples_path)
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, "generated_samples.png not found", ha="center", va="center")
    ax.set_title(LABELS.get(experiment_name, experiment_name))
    ax.axis("off")

plt.tight_layout()

compare_samples_path = os.path.join(COMPARE_OUTPUT_DIR, "samples_comparison.png")
plt.savefig(compare_samples_path, bbox_inches="tight", dpi=150)
plt.show()

print(f"Saved sample comparison to {compare_samples_path}")


Saved sample comparison to /mnt/custom-file-systems/s3/shared/outputs/comparison/samples_comparison.png


# 6. Summary Table

Final-epoch loss, best (minimum) loss reached, and the loss reduction from epoch 1 to the final epoch for each strategy — a quick numeric complement to the plots above.

In [6]:
rows = []
for experiment_name in EXPERIMENTS:
    loss_history = results[experiment_name]["loss_history"]
    first_loss = loss_history[0]
    final_loss = loss_history[-1]
    best_loss = min(loss_history)
    pct_reduction = 100 * (first_loss - final_loss) / first_loss if first_loss else float("nan")

    rows.append({
        "Experiment": LABELS.get(experiment_name, experiment_name),
        "Epochs": len(loss_history),
        "First-epoch loss": round(first_loss, 5),
        "Final-epoch loss": round(final_loss, 5),
        "Best loss": round(best_loss, 5),
        "Loss reduction (%)": round(pct_reduction, 2),
    })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values("Final-epoch loss").reset_index(drop=True)

summary_csv_path = os.path.join(COMPARE_OUTPUT_DIR, "summary.csv")
summary_df.to_csv(summary_csv_path, index=False)

print(f"Saved summary table to {summary_csv_path}")
summary_df


Saved summary table to /mnt/custom-file-systems/s3/shared/outputs/comparison/summary.csv


,Experiment,Epochs,First-epoch loss,Final-epoch loss,Best loss,Loss reduction (%)
0,Backprop + Hebbian,20,0.53511,0.01624,0.01448,96.96
1,Backprop,20,0.54470,0.01646,0.01447,96.98
2,Hebbian only,100,0.98382,0.07858,0.06976,92.01


# 7. Upload Comparison Outputs

Optionally pushes the comparison plots and summary table to S3, under their own `diffusion-hebbian/comparison/` prefix, alongside the three individual experiments.

In [7]:
if S3_BUCKET:
    s3 = boto3.client("s3")
    prefix = "diffusion-hebbian/comparison"

    for local_path in [compare_loss_path, compare_samples_path, summary_csv_path]:
        key = f"{prefix}/{os.path.basename(local_path)}"
        s3.upload_file(local_path, S3_BUCKET, key)
        print(f"Uploaded s3://{S3_BUCKET}/{key}")
else:
    print("S3_BUCKET is None — skipping upload, comparison outputs remain local only.")


S3_BUCKET is None — skipping upload, comparison outputs remain local only.
